In [1]:
import os
os.chdir('..')


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, LogitsProcessor, LogitsProcessorList
import torch
import json
from tqdm import tqdm
import argparse
import re

/raid/home/m13521157/absa-sft-comparison/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
os.environ["CUDA_VISIBLE_DEVICES"] = "5,7"

In [4]:
model_path = "outputs/models/hoasa_hotel/indo/mvp_aos_augment/seed_31415/20251125_213920_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-24890"
tokenizer = AutoTokenizer.from_pretrained(model_path, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.bfloat16, device_map="auto")

`torch_dtype` is deprecated! Use `dtype` instead!


In [9]:
test_data_path = 'dataset/hoasa_hotel/indo/mvp_aos_augment/test.json'
with open(test_data_path, 'r') as f:
	test_data = json.load(f)
test_data[0]

{'sentence_id': 3500,
 'instance_id': 5887,
 'input': 'pelayanan nya sangat ramah . [A] [O] [S]',
 'target': '[A] pelayanan nya [O] sangat ramah [S] positive',
 'element_order': 'aos',
 'task_elements': 'aos',
 'dataset_type': 'hotel_reviews'}

In [12]:
check_inputs = [f'{test_data[i]['input']} =>' for i in range(5)]
check_labels = [f'{test_data[i]["target"]}' for i in range(5)]
check_labels

['[A] pelayanan nya [O] sangat ramah [S] positive',
 '[A] wifi [O]  tidak bagus harus keluar kamar [S] negative',
 '[A] kamarnya [O] beda [S] negative',
 '[A] over all [O] baik [S] positive [SSEP] [A] air hot waternya [O] akan lebih memuaskan jika air hot waternya bisa nyala 24jam [S] negative',
 '[A] fasilatas [O] sesuia [S] positive']

In [40]:
tokenized_inputs = tokenizer(check_inputs, return_tensors='pt', padding=True, truncation=True).to(model.device)

In [ ]:
class BaseConstrainedDecoder(LogitsProcessor):
	'''
	A custom logits processor that modifies the logits during generation.
	All tokens that are not in the input sequence will be masked out (set to -inf) to prevent the model from generating them.
	Exception for special tokens like eos_token, etc.
	Can handle both batched and unbatched input.
	'''
	def __init__(self, input_ids: torch.LongTensor, tokenizer: AutoTokenizer):
		super().__init__()
		self.input_ids = input_ids
		self.tokenizer = tokenizer
		self.special_token_ids = set(tokenizer.all_special_ids)
		self.special_words = set(tokenizer(['positive', 'negative', ' positive', ' negative'], add_special_tokens=False, return_tensors='pt', padding=True, truncation=True)['input_ids'].reshape(-1).tolist())
	
	def _prepare_batch_allowed_tokens(self, tokenizer: AutoTokenizer) -> None:
		
		# Pre-compute allowed token ids for each batch item
		if self.input_ids.dim() == 1:
			# Unbatched: add batch dimension
			self.input_ids = self.input_ids.unsqueeze(0)
		
		# Create a set of allowed tokens for each batch item
		self.batch_allowed_tokens = []
		for i in range(self.input_ids.size(0)):

			# Handle spaced first word
			list_of_tokens = self.input_ids[i].tolist()
			words = tokenizer.decode(list_of_tokens, skip_special_tokens=True)
			words_split = words.split(' ')
			spaced_first_word = f' {words_split[0]}'
			spaced_token_id = set(tokenizer(spaced_first_word, add_special_tokens=False)['input_ids'])

			# Combine all allowed tokens
			allowed_tokens = set(list_of_tokens).union(self.special_token_ids).union(self.special_words).union(spaced_token_id)

			# Store the allowed tokens for this batch item
			self.batch_allowed_tokens.append(allowed_tokens)
	
	def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
		batch_size = scores.size(0)
		
		# Create a mask initialized to -inf for all tokens
		mask = torch.full_like(scores, float('-inf'))
		
		# Apply mask for each batch item
		for i in range(batch_size):
			# Handle case where batch size might be smaller than pre-computed
			allowed_idx = min(i, len(self.batch_allowed_tokens) - 1)
			allowed_token_ids = self.batch_allowed_tokens[allowed_idx]
			
			# Convert allowed token IDs to a list and set their mask values to 0
			allowed_ids_list = list(allowed_token_ids)
			mask[i, allowed_ids_list] = 0.0
		
		# Apply the mask to the scores
		modified_scores = scores + mask
		return modified_scores

class MVPConstrainedDecoder(BaseConstrainedDecoder):
	'''
	A custom logits processor for the MVP dataset that modifies the logits during generation.
	All tokens that are not in the input sequence will be masked out (set to -inf) to prevent the model from generating them.
	Exception for special tokens like eos_token, etc.
	Can handle both batched and unbatched input.
	'''
	def __init__(self, input_ids: torch.LongTensor, tokenizer: AutoTokenizer):
		super().__init__(input_ids, tokenizer)
		self.special_words = self.special_words.union(set(tokenizer([' [SSEP] '], add_special_tokens=False, return_tensors='pt', padding=True, truncation=True)['input_ids'].reshape(-1).tolist()))
		self._prepare_batch_allowed_tokens(tokenizer)

class GASConstrainedDecoder(BaseConstrainedDecoder):
	'''
	A custom logits processor for the GAS dataset that modifies the logits during generation.
	All tokens that are not in the input sequence will be masked out (set to -inf) to prevent the model from generating them.
	Exception for special tokens like eos_token, etc.
	Can handle both batched and unbatched input.
	'''
	def __init__(self, input_ids: torch.LongTensor, tokenizer: AutoTokenizer):
		super().__init__(input_ids, tokenizer)
		self.special_words = self.special_words.union(set(tokenizer(['|' , ' |', '| ', ' | ', ';' , ' ;', '; ', ' ; ', '(', ' (', '( ', ' ( ', ')', ' )', ') ', ' ) '], add_special_tokens=False, return_tensors='pt', padding=True, truncation=True)['input_ids'].reshape(-1).tolist()))
		self._prepare_batch_allowed_tokens(tokenizer)

class LegoABSAConstrainedDecoder(BaseConstrainedDecoder):
	'''
	A custom logits processor for the LegoABSA dataset that modifies the logits during generation.
	All tokens that are not in the input sequence will be masked out (set to -inf) to prevent the model from generating them.
	Exception for special tokens like eos_token, etc.
	Can handle both batched and unbatched input.
	'''
	def __init__(self, input_ids: torch.LongTensor, tokenizer: AutoTokenizer):
		super().__init__(input_ids, tokenizer)
		self.special_words = self.special_words.union(set(tokenizer(['<|aspect|>', '<|opinion|>', '<|sentiment|>', ' ', ';'], add_special_tokens=False, return_tensors='pt', padding=True, truncation=True)['input_ids'].reshape(-1).tolist()))
		self._prepare_batch_allowed_tokens(tokenizer)

In [ ]:
custom_processor = MVPConstrainedDecoder(tokenized_inputs['input_ids'], tokenizer)
processor_list = LogitsProcessorList([custom_processor])
outputs = model.generate(
    **tokenized_inputs,
    max_new_tokens=50,
    # logits_processor=processor_list
)
outputs_logits_processor = model.generate(
	**tokenized_inputs,
	max_new_tokens=50,
	logits_processor=processor_list
)
generated_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)
generated_texts_logits_processor = tokenizer.batch_decode(outputs_logits_processor, skip_special_tokens=True)
assert len(generated_texts) == len(generated_texts_logits_processor)
for i, text in enumerate(generated_texts):
	with_lp = generated_texts_logits_processor[i].split('=>')[-1].strip()
	without_lp = text.split('=>')[-1].strip()
	print(f"Generated {i+1} without logits processor: {without_lp}")
	print(f"Generated {i+1} with logits processor: {with_lp}")
	print(f"Expected label: {check_labels[i]} | with lp: {with_lp == check_labels[i]} | without lp: {without_lp == check_labels[i]}")
	print("-" * 50)

Generated 1 without logits processor: [A] pelayanan nya [O] sangat ramah [S] positive
Generated 1 with logits processor: [A] pelayanan nya [O] sangat ramah [S] positive
Expected label: [A] pelayanan nya [O] sangat ramah [S] positive | with lp: True | without lp: True
--------------------------------------------------
Generated 2 without logits processor: [A] wifi [O] tidak bagus harus keluar kamar [S] negative
Generated 2 with logits processor: [A] wifi [O] tidak bagus harus keluar kamar [S] negative
Expected label: [A] wifi [O]  tidak bagus harus keluar kamar [S] negative | with lp: False | without lp: False
--------------------------------------------------
Generated 3 without logits processor: [A] twin bed [O] twin bed , tetapi yang ada kamarnya beda [S] negative
Generated 3 with logits processor: [A] twin bed [O] twin bed , tetapi yang ada kamarnya beda [S] negative
Expected label: [A] kamarnya [O] beda [S] negative | with lp: False | without lp: False
-----------------------------